In [ ]:
!pip install noaa-sdk

In [ ]:
import requests
import pandas as pd
import numpy as np

lat = 12.9767
lon = 77.5713

PARAMS = ["T2M", "T2M_MAX", "T2M_MIN", "RH2M", "WS2M", "PRECTOT", "PS"]

def get_year_data(year):
    url = (
        "https://power.larc.nasa.gov/api/temporal/daily/point"
        f"?start={year}0101&end={year}1231"
        f"&latitude={lat}&longitude={lon}"
        f"&parameters={','.join(PARAMS)}"
        "&format=JSON&community=AG"
    )

    r = requests.get(url)
    data = r.json()

    params = data["properties"]["parameter"]

    # Get list of dates from ANY available parameter
    any_param = list(params.values())[0]
    dates = list(any_param.keys())

    df = pd.DataFrame({"date": dates})

    # Fill each parameter safely
    for p in PARAMS:
        if p in params:
            df[p] = list(params[p].values())
        else:
            print(f"⚠️ Warning: {p} missing for {year}. Filling with NaN.")
            df[p] = np.nan

    return df


all_data = []

for year in range(2016, 2020):
    print(f"\n📅 Fetching {year}...")
    df = get_year_data(year)
    print("Rows:", len(df))
    all_data.append(df)

# Combine all years
final = pd.concat(all_data, ignore_index=True)

# Sort by date
final = final.sort_values("date")

final.to_csv("bengaluru_weather_nasa_2016_2019.csv", index=False)

print("\n✅ Saved bengaluru_weather_nasa_2016_2019.csv")



📅 Fetching 2016...
⚠️ Warning: PRECTOT missing for 2016. Filling with NaN.
Rows: 366

📅 Fetching 2017...
⚠️ Warning: PRECTOT missing for 2017. Filling with NaN.
Rows: 365

📅 Fetching 2018...
⚠️ Warning: PRECTOT missing for 2018. Filling with NaN.
Rows: 365

📅 Fetching 2019...
⚠️ Warning: PRECTOT missing for 2019. Filling with NaN.
Rows: 365

✅ Saved bengaluru_weather_nasa_2016_2019.csv


The precipitation values in the NASA POWER dataset were missing for most days in the 2016–2019 period. This happens because NASA’s PRECTOT parameter is model-derived, and at fine spatial scales (city-level coordinates like Majestic, Bengaluru), it often returns incomplete or blank records.

Since precipitation is an essential meteorological confounder in air-pollution analysis, we could not rely on the NASA dataset alone. Therefore, we extracted accurate daily rainfall records from the India Meteorological Department (IMD) dataset available through:

👉 https://imdpune.gov.in/lrfindex.php

IMD provides ground-observed, station-level rainfall measurements, which are more reliable for regional pollution modelling. We downloaded the IMD gridded rainfall dataset (0.25° × 0.25° resolution) and extracted the grid cell corresponding to the Majestic area.

In [ ]:
import xarray as xr
import pandas as pd

# Majestic coordinates
target_lat = 12.9767
target_lon = 77.5713

def extract_rainfall(file_path):
    """
    Extract rainfall for the IMD grid cell closest to Majestic.
    Returns a dataframe with columns: ['date', 'rainfall_mm']
    """
    ds = xr.open_dataset(file_path)

    # Grid values
    latitudes = ds["LATITUDE"].values
    longitudes = ds["LONGITUDE"].values

    # Nearest grid point (0.25° × 0.25° IMD grid)
    nearest_lat = latitudes[(abs(latitudes - target_lat)).argmin()]
    nearest_lon = longitudes[(abs(longitudes - target_lon)).argmin()]

    # Extract rainfall at this grid cell
    rain = ds["RAINFALL"].sel(LATITUDE=nearest_lat, LONGITUDE=nearest_lon)

    # Convert to a dataframe
    df = rain.to_dataframe().reset_index()

    # Rename
    df = df.rename(columns={
        "TIME": "date",
        "RAINFALL": "rainfall_mm"
    })

    # Keep only needed columns
    return df[["date", "rainfall_mm"]]


# ---- Extract each year ----
df_2016 = extract_rainfall("/content/RF25_ind2016_rfp25.nc")
df_2017 = extract_rainfall("/content/RF25_ind2017_rfp25.nc")
df_2018 = extract_rainfall("/content/RF25_ind2018_rfp25.nc")
df_2019 = extract_rainfall("/content/RF25_ind2019_rfp25.nc")

# ---- Combine all years ----
rain_all = pd.concat([df_2016, df_2017, df_2018, df_2019], ignore_index=True)

# ---- Save combined rainfall file ----
rain_all.to_csv("bengaluru_imd_rainfall_2016_2019.csv", index=False)

In [ ]:
import pandas as pd

# Load both datasets from Colab disk
weather = pd.read_csv("/content/bengaluru_weather_clean.csv")
rain = pd.read_csv("/content/bengaluru_imd_rainfall_2016_2019.csv")

weather["date"] = pd.to_datetime(weather["date"], format="%Y-%m-%d", errors="coerce")

mask_numeric = weather["date"].isna()
if mask_numeric.any():
    weather.loc[mask_numeric, "date"] = pd.to_datetime(
        weather.loc[mask_numeric, "date"].astype(str),
        format="%Y%m%d",
        errors="coerce"
    )

# CLEAN NASA FILE COLUMN NAMES (if required)
weather = weather.rename(columns={
    "T2M": "temp_mean",
    "T2M_MAX": "temp_max",
    "T2M_MIN": "temp_min",
    "RH2M": "humidity",
    "WS2M": "wind_speed",
    "PRECTOT": "precipitation",
    "PS": "pressure"
})

# PREP IMD RAINFALL FILE
rain["date"] = pd.to_datetime(rain["date"])
rain = rain.rename(columns={"rainfall_mm": "rainfall_imd"})

# MERGE NASA + IMD RAINFALL
merged = weather.merge(rain, on="date", how="left")

# REPLACE NASA precipitation with IMD rainfall
merged["precipitation"] = merged["rainfall_imd"]

# Remove old IMD column
merged = merged.drop(columns=["rainfall_imd"])

# REMOVE ANY EMPTY / USELESS COLUMNS
merged = merged.dropna(axis=1, how="all")

# SORT + SAVE FINAL FILE
merged = merged.sort_values("date").reset_index(drop=True)

output_path = "/content/bengaluru_weather_final.csv"
merged.to_csv(output_path, index=False)

print("DONE! Saved as:", output_path)
print("Rows:", len(merged))
print("Columns:", merged.columns.tolist())


DONE! Saved as: /content/bengaluru_weather_final.csv
Rows: 1461
Columns: ['date', 'temp_mean', 'temp_max', 'temp_min', 'humidity', 'wind_speed', 'precipitation', 'pressure']
